# IBGE Municipalities - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import col, lower, translate

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.ibge_municipalities"
target_table = f"{catalog}.silver.ibge_municipalities"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
silver_df = bronze_df.withColumn(
    "municipality_name_normalized",
    translate(
        lower(col("municipality_name")),
        "áàâãäéèêëíìîïóòôõöúùûüç-'",
        "aaaaaeeeeiiiiooooouuuuc  "
    )
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)